# Analysis guide

This notebook follows the paper section by section. For each analysis it lists the scripts, in the order they were
run, and loads the deposited result file that holds the numbers the paper reports. It reads only files in this
repository and needs only pandas. Run it from the repository root.

The scripts use paths under the original analysis folder, shown as `/path/to/revision`: set it to the repository root.
The network files that the scripts read from `data/` are in `data/network/`. `scripts/` holds the main analysis grouped by
analysis and `analyses/` the self-contained later analyses; each folder has a README.

Patient-level data (TCGA, METABRIC) and licensed or third-party downloads are not included; `README.md` lists them.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1. Network assembly (Methods 2.1; Results 3.1)

Scripts, in run order: `scripts/01_network_assembly/02_build_canonical_network.py` → `03_layer_TF_target.R` (TRRUST v2) →
`04_layer_TF_miRNA.R` (TransmiR v2.0) → `05_layer_miRNA_miRNA.R` (co-transcribed pairs) →
`06a_string_map_and_fetch.R`, `06_layer_gene_gene.R` (STRING v12) → `07_edge_evidence_tier.R` (multiMiR evidence
tiers) → `scripts/15_figures_and_tables/21_tables.R` (Table S2).

Table S2 lists every interaction. The analysed network excludes the 30 miRNA–miRNA edges of the exemplar circuits
(`in_analysed_network`). Literature signs are in `sign`, `trrust_mode` and `transmir_mode`; the `sign` column of
`data/network/canonical_edges.tsv` is the original deposit's placeholder.

In [ ]:
nodes = pd.read_csv("data/network/canonical_nodes.tsv", sep="\t")
edges = pd.read_csv("supplementary_tables/TableS2_all_interactions.csv", low_memory=False)
analysed = edges[edges.in_analysed_network]
print(len(nodes), "nodes:", nodes.type.value_counts().to_dict())
print(len(analysed), "edges in the analysed network:", analysed.edge_type.value_counts().to_dict())
print("sign source:", analysed.sign_source.value_counts().to_dict())
tiers = analysed[analysed.edge_type == "miRNA_target"].tier.value_counts()
print("miRNA-target evidence tiers:", tiers.to_dict(), (100 * tiers / tiers.sum()).round(1).to_dict())

587 nodes: {'miRNA': 223, 'Gene': 207, 'TF': 157}
6829 edges in the analysed network: {'miRNA_target': 4819, 'TF_miRNA': 1239, 'TF_target': 770, 'gene_gene': 1}
sign source: {'mechanistic (miRNA repression)': 4819, 'TransmiR': 1106, 'unannotated (excluded from sign-dependent analyses)': 467, 'TRRUST': 437}
miRNA-target evidence tiers: {'weak': 2309, 'predicted_only': 1803, 'strong': 707} {'weak': 47.9, 'predicted_only': 37.4, 'strong': 14.7}


## 2. The miRNA–TF pair filter (Methods 2.1; Supplementary Note S1)

The filter was computed in the authors' spreadsheet (`data/pair_filter/`; see its `README.md`): a lower-tail
cumulative hypergeometric *P* for the targets each miRNA–TF pair shares in a 256-gene universe, with
Benjamini–Hochberg control over the 2,676 rows. It was not run from a script. These scripts recompute it and compare
it with the network: `analyses/six_node_pattern/E/E5/e5a_hypergeometric_filter.py`, `analyses/six_node_followups/Q123/`,
`Q5/` and `Q9/`.

The cell below recomputes the filter from its input block and reproduces the workbook's retained pairs.

In [ ]:
from math import comb

def hypergeom_p(m, t, i, upper):
    """P(X <= i) (lower tail) or P(X >= i) (upper tail) for m miRNA targets and t TF targets among 256 genes."""
    if t > 256:
        return float("nan")
    xs = range(i, t + 1) if upper else range(i + 1)
    return sum(comb(m, x) * comb(256 - m, t - x) for x in xs) / comb(256, t)

v = pd.read_csv("data/pair_filter/value.txt", sep="\t", header=None, names=["mirna", "tf", "m", "t", "i"])
v["p_lower"] = [hypergeom_p(m, t, i, False) for m, t, i in zip(v.m, v.t, v.i)]
v["p_upper"] = [hypergeom_p(m, t, i, True) for m, t, i in zip(v.m, v.t, v.i)]
rank = v.p_lower.rank(method="first")
v["retained"] = rank <= rank[v.p_lower <= rank / len(v) * 0.05].max()        # Benjamini-Hochberg, 5%
pairs = v.drop_duplicates(["mirna", "tf"]); kept = pairs[pairs.retained]
print(len(v), "rows;", len(pairs), "distinct pairs;", len(kept), "retained pairs, of which",
      (kept.i == 0).sum(), "share no target")
print("pairs with a defined upper-tail P:", pairs.p_upper.notna().sum(), "| upper-tail P < 0.05:", (pairs.p_upper < 0.05).sum())

2676 rows; 2253 distinct pairs; 1566 retained pairs, of which 1547 share no target
pairs with a defined upper-tail P: 2111 | upper-tail P < 0.05: 0


## 3. Three-node feed-forward loops and their coherence (Methods 2.2; Results 3.2)

Scripts: `scripts/03_ffl_census/ffl_def.py`, `ffl_cores.py`, `ffl_enum.c` (enumeration) → `scripts/03_ffl_census/01a_counting_convention.py` →
`scripts/03_ffl_census/01b_coherence_corrected.py` (sign-resolved coherence, Mangan–Alon sub-types) → `scripts/15_figures_and_tables/21_tables.R`
(Table S4). The count of all three-node FFLs in the analysed network is the observed row of the null runs in
section 5.

In [ ]:
cores = pd.read_csv("supplementary_tables/TableS4_ffl_cores.csv")
resolved = cores[cores.coherence != "unresolved"]
coherent = resolved.coherence.str.startswith("C")
print(len(cores), "typed cores:", cores["class"].value_counts().to_dict())
print(len(resolved), "sign-resolved cores:", f"{100 * coherent.mean():.1f}% coherent;", resolved.coherence.value_counts().to_dict())
print("coherent by class (%):", (100 * coherent.groupby(resolved["class"]).mean()).round(1).to_dict())

1649 typed cores: {'Composite-FFL': 1434, 'miRNA-FFL': 206, 'TF-FFL': 9}
898 sign-resolved cores: 43.8% coherent; {'I1': 364, 'C4': 180, 'I2': 141, 'C3': 112, 'C2': 101}
coherent by class (%): {'Composite-FFL': 37.9, 'TF-FFL': 40.0, 'miRNA-FFL': 78.9}


## 4. Census of four- to seven-node modules (Results 3.2; Fig. 1)

Scripts: `scripts/03_ffl_census/03_ffl_census.py` and `scripts/03_ffl_census/03_ffl_census_nolegacymirna.py` (graph building) →
`analyses/census_and_motif_nulls/graphs/make_graphs.py` → `run_all.sh`, which compiles and runs
`census/ffl_census_composition.c` and merges its partitions with `census/aggregate_census.py` →
`census/summarise_census.py`, which writes `census_all_graphs.csv` (copied to `results/v2/census_rerun/`).

Graph `nolegacy` is the census graph of the paper, `nolegacy_nostring` the same graph without STRING arcs and `table3`
the graph that still holds the 30 exemplar miRNA–miRNA edges. The seven-node counts are sampled estimates. Older
census files in `results/` (`ffl_census_summary.csv`, `ffl_census.csv`, `ffl_higher_order.csv.gz`) come from the
first round, on the graph with the exemplar edges.

In [ ]:
census = pd.read_csv("results/v2/census_rerun/census_all_graphs.csv")
wide = census[census.graph.isin(["nolegacy", "nolegacy_nostring", "table3"])].pivot(
    index="n", columns="graph", values=["modules", "max_edge_classes"])
print(wide.round(0).to_string())
modules = wide["modules"].astype(float)
print("% of modules lost without STRING arcs:", (100 * (1 - modules.nolegacy_nostring / modules.nolegacy)).round(1).to_dict())
print("% of modules due to the 30 exemplar edges:", (100 * (1 - modules.nolegacy / modules.table3)).round(1).to_dict())

           modules                                max_edge_classes                         
graph     nolegacy nolegacy_nostring       table3         nolegacy nolegacy_nostring table3
n                                                                                          
3           5833.0            3432.0       6037.0              3.0               3.0    3.0
4         116505.0           81104.0     117967.0              5.0               5.0    5.0
5        1506263.0          791085.0    1534131.0              6.0               5.0    6.0
6       19401840.0         8751860.0   19838598.0              6.0               6.0    7.0
7      225761602.0        87313549.0  232029403.0              6.0               6.0    7.0
% of modules lost without STRING arcs: {3: 41.2, 4: 30.4, 5: 47.5, 6: 54.9, 7: 61.3}
% of modules due to the 30 exemplar edges: {3: 3.4, 4: 1.2, 5: 1.8, 6: 2.2, 7: 2.7}


## 5. Motif significance of three-node FFLs (Methods 2.2; Results 3.2)

Scripts: `scripts/03_ffl_census/v2_graph_final.py`, `v2_export_graph.py` → `analyses/census_and_motif_nulls/graphs/make_graphs.py`
→ `run_all.sh`, which runs `nulls/v2_null.c` (1,000 randomisations per null model) → `nulls/summarise_nulls.py`
(`motif_nulls_all_graphs.csv`, copied to `results/v2/census_rerun/`). `dep_nolegacy` is the analysed
network and `pub_nolegacy` the census graph. The re-typed variant (22 co-regulators typed as genes) is
`analyses/six_node_pattern/E/E5/` (`e5bc_retype_graphs.py` → `e5b_null_summary.py`).

In [ ]:
folder = "analyses/census_and_motif_nulls/nulls/"
rows = []
for run in ["A_dep", "B_dep", "C_dep", "B_pub", "C_pub"]:
    t = pd.read_csv(f"{folder}null_{run}_nolegacy.tsv", sep="\t", index_col=0)
    observed, null = t.loc["obs"], t.drop("obs")
    for motif in ["total", "comp"]:
        mean, sd = null[motif].mean(), null[motif].std()
        rows.append([run, motif, observed[motif], round(mean), f"{100 * (observed[motif] / mean - 1):+.1f}%",
                     round((observed[motif] - mean) / sd, 1),
                     round((1 + (null[motif] >= observed[motif]).sum()) / (len(null) + 1), 3),
                     round((1 + (null[motif] <= observed[motif]).sum()) / (len(null) + 1), 3)])
print(pd.DataFrame(rows, columns=["null", "motif", "observed", "null mean", "excess", "Z", "p (enriched)", "p (depleted)"]).to_string(index=False))

 null motif  observed  null mean  excess     Z  p (enriched)  p (depleted)
A_dep total      1774       2638  -32.8% -10.7         1.000         0.001
A_dep  comp      1296        399 +224.9%  26.3         0.001         1.000
B_dep total      1774       1644   +7.9%   2.4         0.013         0.988
B_dep  comp      1296       1217   +6.5%   1.8         0.043         0.959
C_dep total      1774       1595  +11.2%   3.2         0.001         1.000
C_dep  comp      1296       1202   +7.9%   2.1         0.027         0.975
B_pub total      5833       6778  -13.9%  -5.0         1.000         0.001
B_pub  comp      1375       1538  -10.6%  -3.1         1.000         0.001
C_pub total      5833       6506  -10.3%  -3.5         1.000         0.001
C_pub  comp      1375       1419   -3.1%  -0.8         0.777         0.232


## 6. The six-node composite pattern (Results 3.2; Supplementary Note S2)

Scripts: `analyses/six_node_pattern/S1/make_s1_inputs.py` → `S1/s1_null6.c` (run as listed in `S1/jobs.txt`) →
`S1/s1_summarise.py`, `S1/s1_cotranscription_overlap.py`. Variant graphs: `S7/s7b_restricted_gene_gene.py`
(physical STRING links), `S7/s7c_build_validated_graphs.py` (validated miRNA targets), `E/E5/e5bc_retype_graphs.py`
→ `analyses/six_node_followups/Q6/` (`q6_make_graphs.py` → `run_q6.sh` → `q6_summarise.py`). Clusters:
`analyses/six_node_followups/Q7/q7_cotranscription.py`. The instance lists come from
`analyses/six_node_pattern_networks/build_bhat_networks.py`.

"Over-represented" requires *p* < 0.05 under both NULL-C (reciprocity and class preserved) and NULL-L (analysed network
fixed, census layers randomised).

In [ ]:
columns = ["observed", "NULL_C_ratio", "NULL_C_p_upper", "NULL_L_ratio", "NULL_L_p_upper"]
s1 = pd.read_csv("analyses/six_node_pattern/S1/s1_decision.csv").set_index("pattern")
print(s1.loc[["BHAT6 composite, instances", "CENSUS6 (c)-with-core modules"], columns].round(3).to_string())
q6 = pd.read_csv("analyses/six_node_followups/Q6/q6_decision.csv").query("pattern == 'BHAT6 composite, instances'")
print(q6.set_index("graph")[columns].round(3).to_string())
clusters = pd.read_csv("analyses/six_node_followups/Q7/q7c_bhat6_by_genomic_cluster.csv").head(5)
print("five largest clusters:", clusters.instances.tolist(), "=", clusters.instances.sum(), "of 3,898 instances",
      f"({100 * clusters.instances.sum() / 3898:.1f}%)")

                               observed  NULL_C_ratio  NULL_C_p_upper  NULL_L_ratio  NULL_L_p_upper
pattern                                                                                            
BHAT6 composite, instances       3898.0         2.347           0.002         3.688           0.001
CENSUS6 (c)-with-core modules  210115.0         1.375           0.078         0.832           0.846
             observed  NULL_C_ratio  NULL_C_p_upper  NULL_L_ratio  NULL_L_p_upper
graph                                                                            
retyped22      5532.0         2.472           0.001         4.736           0.001
validated      3012.0         7.538           0.001        11.787           0.001
physical900     906.0         1.436           0.109         2.614           0.002
five largest clusters: [1436, 497, 282, 273, 243] = 2731 of 3,898 instances (70.1%)


## 7. Node prioritisation (Methods 2.3; Results 3.3)

Scripts: `scripts/10_survival_and_clinical/cox_models/10_survival_cox_hubs.R` (topology) and the expression, multi-omics and DepMap scripts listed in the
header of `scripts/06_node_prioritisation/01_prioritisation.R` → `01_prioritisation.R` → `45_pubmed_all_nodes.py` →
`47_literature_vs_topology.py` (all in `scripts/06_node_prioritisation/`). `analyses/six_node_pattern/F/F1/` re-runs the ranking on the analysed network.

Ranks are recomputed from the `priority` score of Table S1. The rank columns of `results/v5/node_prioritisation_full.csv`
count the four unrankable miRNAs first and are four places too low.

In [ ]:
s1 = pd.read_csv("supplementary_tables/TableS1_node_prioritisation_full.csv").dropna(subset=["priority"])
s1["rank"] = s1.priority.rank(ascending=False).astype(int)
s1["rank_in_class"] = s1.groupby("type").priority.rank(ascending=False).astype(int)
print(len(s1), "ranked nodes:", s1.type.value_counts().to_dict())
print(s1.set_index("name").loc[["SP1", "RELA", "NFKB1", "E2F1", "hsa-miR-383", "EGR2"], ["ffl_cores", "rank", "rank_in_class"]].to_string())
lit = pd.read_csv("results/v5/network_literature_vs_topology_stats.csv")
print(lit.query("scope == 'all nodes' and literature == 'pubmed_total'")[["metric", "rho", "p"]].to_string(index=False))

583 ranked nodes: {'miRNA': 219, 'Gene': 207, 'TF': 157}
             ffl_cores  rank  rank_in_class
name                                       
SP1                226   227             66
RELA               201   315             86
NFKB1              179   410            108
E2F1                66     2              1
hsa-miR-383          6     8              4
EGR2                 0    20              6
         metric      rho            p
         degree 0.245202 1.741530e-09
betw_undirected 0.242103 2.819096e-09
      ffl_cores 0.313344 7.718382e-15


## 8. Expression validation and replication (Methods 2.4; Results 3.4)

Scripts: `scripts/07_expression_validation/tcga_differential_expression/01_tcga_brca_prep_de.R`, `01b_fix_entrez_to_symbol.R`, `01b_fix_gene_de.R` → `scripts/07_expression_validation/tcga_differential_expression/01_de_limma.R`
(differential expression) → `scripts/07_expression_validation/edge_correlations/08_expression_validation.R` (sign concordance against matched nulls) →
`scripts/07_expression_validation/edge_correlations/09_anova_ffl_named_axes.R` → `scripts/07_expression_validation/external_cohorts/16_external_geo_de.R` (GEO replication). Protein level (CPTAC, 101
tumours): `scripts/07_expression_validation/protein_cptac_and_rppa/20_cptac_prep.R` → `21_cptac_protein_validation.R` → `22_cptac_axes_mrnaprot_phospho.R`, and
`23_rppa_second_platform.R` (RPPA). These steps need TCGA and CPTAC patient-level data, which are not redistributed;
their outputs are deposited.

In [ ]:
for f in ["results/v2/v2_DE_genes.csv", "results/v2/v2_DE_mirnas.csv"]:
    de = pd.read_csv(f)
    sig = de[(de["adj.P.Val"] < 0.05) & (de.logFC.abs() > 1)]
    print(f, "- tested", len(de), "| differential", len(sig), "| up", (sig.logFC > 0).sum(), "| down", (sig.logFC < 0).sum())
conc = pd.read_csv("results/sign_concordance_summary.csv")
print(conc[["stratum", "n_edges_tested", "concordance_rate", "null_concordance_rate", "binom_p_vs_null"]].to_string(index=False, float_format="%.3g"))
cptac = pd.read_csv("results/multiomics/cptac_concordance_summary.csv")
cptac = cptac[cptac.stratum.isin(["TF_target_Activation", "miRNA_target_ALL"])]
print("CPTAC:"); print(cptac[["stratum", "assay", "n_edges_conc_tested", "concordance_rate", "null_concordance_rate"]].round(3).to_string(index=False))

results/v2/v2_DE_genes.csv - tested 20250 | differential 4231 | up 1416 | down 2815
results/v2/v2_DE_mirnas.csv - tested 495 | differential 80 | up 38 | down 42
              stratum  n_edges_tested  concordance_rate  null_concordance_rate  binom_p_vs_null
   ALL_sign_annotated            6147             0.499                  0.494            0.218
     miRNA_target_ALL            4628             0.479                  0.483             0.71
  miRNA_target_strong             701             0.495                   0.51            0.795
    miRNA_target_weak            2294             0.527                  0.499          0.00441
miRNA_target_predonly            1633             0.404                  0.448                1
 TF_target_Activation             301             0.711                  0.603         5.85e-05
 TF_target_Repression             136             0.426                  0.407            0.349
  TF_miRNA_Activation             708             0.578                

## 9. Stromal mediation and compartment tests (Methods 2.6; Results 3.5)

Mediation across 42 estimates of stromal content: `scripts/08_stroma_and_cell_types/deconvolution_and_mediation/20_deconv_install.R` → `22_deconv_prep.R` →
`23_cibersort_impl.R`, `24_wu_signature_matrix.R` → `25_run_deconvolution.R` → `26_instaprism_bayesprism.R` →
`27_assemble_scores.R` → `34_collagenfree_variants.R` → `35_consensus_mediator.R` → `42_deconv2_external_refs.R` →
`43_deconv2_newmethods.R` → `52_deconv2_methylation_epidish.R` → `45_deconv2_assemble.R` → `46_deconv2_mediation.R`;
the 42-estimate counts are `analyses/six_node_pattern/E/E7/e7_mediation_intervals.py`.
Designs without deconvolution: `scripts/08_stroma_and_cell_types/spatial/05_spatial_full.py` → `06_spatial_mediation_summary.py` (Visium);
`scripts/08_stroma_and_cell_types/single_cell/ct_02_scrna_extract.sh` → `ct_02b_pseudobulk_all.sh` (single-cell atlas); `scripts/08_stroma_and_cell_types/xenografts/01_fetch_pdmr.py` →
`03_pdx_analysis.py` → `04_pdx_foldchange_and_caf.py` (tumour–xenograft pairs). miR-29 and the collagens at the
protein level: `scripts/07_expression_validation/protein_cptac_and_rppa/22_cptac_axes_mrnaprot_phospho.R`.

In [ ]:
e7 = pd.read_csv("analyses/six_node_pattern/E/E7/e7_inconsistent_counts.csv")
print(e7[e7.estimate_set.str.startswith("42")][["arm", "n_estimates", "n_inconsistent"]].to_string(index=False))
spots = pd.read_csv("results/v6/spatial_bulk_vs_compartment_summary.csv").query("outcome == 'COL1A1'")
print(spots[["scheme", "mean_abs_rho_tcga_bulk", "mean_abs_rho_stromal", "pct_of_bulk_retained_in_stroma", "wilcoxon_p_stromal_vs_bulk"]].round(4).to_string(index=False))
atlas = pd.read_csv("results/v6/pdx_wu_atlas_caf_ratio_allgenes.csv", index_col=0).loc[["COL1A1", "COL3A1"]]
print("CAF / carcinoma expression in the single-cell atlas:", (atlas["CAFs"] / atlas["Cancer Epithelial"]).round(1).to_dict())
pdx = pd.read_csv("results/v6/pdx_stromality_vs_rankdrop.csv")
print("xenografts, 40 genes: Spearman rho, median fold change on engraftment vs CAF/carcinoma ratio =",
      round(pdx[["median_log2FC_panelnormalised", "log2_CAF_over_CancerEpi"]].corr("spearman").iloc[0, 1], 3))
axes = pd.read_csv("results/multiomics/cptac_named_axes_protein.csv")
mir29 = axes[axes.feature_a.str.startswith("hsa-miR-29") & axes.layer.isin(["miRNA_vs_mRNA", "miRNA_vs_protein"])]
mir29 = mir29.assign(collagen=mir29.feature_b.str.split("_").str[0]).pivot_table(
    index=["feature_a", "collagen"], columns="layer", values="rho")
print(mir29.round(3).to_string())
print("stronger at the protein level:", (mir29.miRNA_vs_protein.abs() > mir29.miRNA_vs_mRNA.abs()).sum(), "of", len(mir29))

                  arm  n_estimates  n_inconsistent
       ETS1 -> COL1A1           42              23
      NFKB1 -> COL1A1           42              32
       RELA -> COL1A1           42              22
        SP1 -> COL1A1           42              38
hsa-miR-29a -> COL1A1           42               6
hsa-miR-29b -> COL1A1           42               3
hsa-miR-29c -> COL1A1           42               7
           scheme  mean_abs_rho_tcga_bulk  mean_abs_rho_stromal  pct_of_bulk_retained_in_stroma  wilcoxon_p_stromal_vs_bulk
author_annotation                  0.1884                0.0888                         47.1043                      0.0425
   marker_tertile                  0.1884                0.0218                         11.5488                      0.0010
CAF / carcinoma expression in the single-cell atlas: {'COL1A1': 146.1, 'COL3A1': 160.7}
xenografts, 40 genes: Spearman rho, median fold change on engraftment vs CAF/carcinoma ratio = -0.887
layer                 miRNA_vs

## 10. Regulatory DNA at the collagen promoters (Results 3.5; Supplementary Note S1)

Enformer: `scripts/09_regulatory_evidence/sequence_models/20_seqreg_regions.py` → `28_seqreg_motifscan.py` → `22b_seqreg_enformer_ism.py` →
`41_seqreg_ext_occlusion_null.py` → `44_seqreg_ext_synthesis.py`, with `45_seqreg_ext_splicedonor_control.py`.
Borzoi: `analyses/perturbation_tests/P4/` (`p4_download_weights.py` → `p4_borzoi_occlusion.py` → `p4_summarise.py`).
Accessibility in ENCODE fibroblast and epithelial samples: `scripts/09_regulatory_evidence/encode_accessibility/atac_04_encode_fetch.py` → `atac_05_accessibility.py`
→ `atac_06_encode_compartment.py` → `atac_compartment_specificity_tests_rebuild.py` (Fisher tests; see `README.md`).

In [ ]:
occl = pd.read_csv("results/v3/seqreg_ext_occlusion_calibrated.csv")
print(occl[["region", "family", "rank", "n_sites_total", "best_pct_change"]].round(2).to_string(index=False))
donor = pd.read_csv("results/v3/seqreg_ext_splicedonor_control.csv")
print(donor[["region", "variant", "pct_change_fib_cage"]].round(1).to_string(index=False))
for line in open("analyses/perturbation_tests/P4/p4_summary.txt"):
    if "decision" in line:
        print(line.strip())
atac = pd.read_csv("results/v6/atac_encode_compartment_specificity_tests.csv")
print(atac[atac.gene.isin(["COL1A1", "COL3A1"])].to_string(index=False, float_format="%.4g"))

region family  rank  n_sites_total  best_pct_change
COL1A1  NF-kB 133.0            618            -0.63
COL1A1     SP  33.0            618            -3.29
COL1A1    ETS 201.0            618            -0.23
COL3A1  NF-kB 184.0           1008            -2.08
COL3A1     SP   NaN           1008              NaN
COL3A1    ETS  10.0           1008           -28.82
region                    variant  pct_change_fib_cage
COL3A1        donor_kill_GT_to_AA                -27.3
COL3A1        donor_kill_GT_to_CC                -28.5
COL3A1 ets_core_kill_GGAA_to_CCTT                 -8.2
COL3A1 ets_core_kill_GGAA_to_TTTT                 -5.0
COL3A1         exonic_control_4bp                  0.7
COL1A1        donor_kill_GT_to_AA                 -9.2
COL1A1        donor_kill_GT_to_CC                -10.3
COL1A1         exonic_control_4bp                  0.2
Enformer (deposited, fibroblast CAGE) decision: REPLICATED (criteria 111)
Borzoi (4 replicates, fibroblast RNA-seq exon sum) decision: REPLIC

## 11. The miR-29–collagen axis and clinical association (Results 3.6)

Survival meta-analysis: `scripts/10_survival_and_clinical/multi_cohort/M3_build_mirna_cohorts.R` → `scripts/10_survival_and_clinical/mirna_survival_meta_analysis/c01_geo_search.py` … `c10_dump_full_matrices.py`
→ `c11_mirna_cox_meta.R` → `c24_sensitivity.R`; Benjamini–Hochberg over the eleven reported miRNAs:
`analyses/six_node_pattern/G/g1_bh_eleven_mirnas.py` (Table S8).
Reactive stroma: `scripts/10_survival_and_clinical/reactive_stroma_and_neoadjuvant/40_farmer_extract_sources.py` → `41_build_signature_collection.R` → `42_farmer_scores_tcga.R`
→ `43_farmer_module_overlap.R` → `44_farmer_mediation.R` → `46_neoadjuvant_response.R` → `49_neoadj_meanz_sensitivity.R`.

In [ ]:
s8 = pd.read_csv("supplementary_tables/TableS8_mirna_survival_meta.csv")
pooled = s8[s8.row_type.eq("RE_pooled") & s8.miRNA.isin(["miR-29a", "miR-195", "miR-204"])]
print(pooled[["miRNA", "k", "n", "nevent", "HR", "lo", "hi", "p", "I2", "q_BH"]].to_string(index=False, float_format="%.3g"))
overlap = pd.read_csv("results/v5/farmer_module_overlap.csv").query("module == 'FFL_6node_all' and signature == 'FARMER_STROMAL'")
print(overlap[["n_module", "n_overlap", "fold_enrichment", "p_hypergeometric"]].to_string(index=False))
pcr = pd.read_csv("results/v5/farmer_neoadjuvant_meta.csv").set_index("score")
print("collagen pair, odds ratio for pCR per SD:", pcr.loc["COLLAGEN_PAIR", ["OR_random", "p_random", "I2"]].round(3).to_dict())

  miRNA  k    n  nevent    HR    lo    hi        p   I2    q_BH
miR-195  6 1732     483 0.833 0.753 0.923 0.000444 20.7 0.00244
miR-204  6 1691     464 0.795 0.663 0.953   0.0131 68.7   0.048
miR-29a  6 1732     483 0.826 0.758   0.9 1.36e-05    0 0.00015
 n_module  n_overlap  fold_enrichment  p_hypergeometric
      307         13            17.15      3.728410e-13
collagen pair, odds ratio for pCR per SD: {'OR_random': 0.846, 'p_random': 0.016, 'I2': 16.834}


## 12. Dynamics of higher-order modules (Methods 2.5; Results 3.7)

Scripts: `scripts/11_dynamics/dyn_models.py`, `dyn_sample.py`, `dyn_metrics.py` (models and sampling) →
`03_higher_order_sweep.py` → `03d_higher_order_compI1.py` → `03e_higher_order_convergence_audit.py` →
`03f_n6_limit_cycle_certification.py`; four-node control:
`analyses/dynamics_controls/c3_tf2_only_control.py`; layer and sign variants:
`analyses/six_node_pattern/S2/` and `S6/` (`*_run_modules.py` → `s2_certify_limit_cycles.py` → `s2_analyse.py`) and
`S4/s4_damped_oscillation_ratio.py`. `COMP_C2_toggle` is the composite module (its core is coherent type 4 in the
Mangan–Alon numbering; see the note at the top of `dyn_models.py`).

In [ ]:
gain = pd.read_csv("results/v3/dynamics_higher_order_gain_loss.csv")
gain["net"] = gain.pct_gain - gain.pct_loss
behaviours = ["is_memory", "is_ultrasensitive", "is_pulse", "is_damped_osc", "is_sustained_osc"]
print(gain[gain.module.isin(["n4", "n5", "n6"]) & gain.behaviour.isin(behaviours)].pivot_table(
    index="behaviour", columns=["family", "module"], values="net").round(1).to_string())
six = pd.read_csv("results/v3/dynamics_higher_order_persetset.csv.gz").query("family == 'COMP_C2_toggle' and module == 'n6'")
print("six-node module, % of parameter sets:", six[["is_pulse", "is_damped_osc", "is_sustained_osc"]].mean().mul(100).round(2).to_dict())
certified = pd.read_csv("results/v3/dynamics_n6_limit_cycle_certified.csv").certified_limit_cycle.sum()
control = pd.read_csv("analyses/dynamics_controls/c3_persetset.csv.gz").query("module == 'n4tf'")
print("certified limit cycles:", int(certified), "| four-node control, %:", control[["is_pulse", "is_damped_osc", "is_sustained_osc"]].mean().mul(100).round(2).to_dict())

family            COMP_C2_toggle             I1_miRNA_FFL            
module                        n4    n5    n6           n4    n5    n6
behaviour                                                            
is_damped_osc                0.0   0.0   5.3          0.0   0.0   0.0
is_memory                   34.3  23.8  33.8         32.0  25.9  22.9
is_pulse                    -0.0  -0.0  25.1         -7.1  -2.7   2.4
is_sustained_osc             0.0   0.0   0.7          0.0   0.0   0.0
is_ultrasensitive           -9.3 -18.3   5.2        -14.9 -11.3  -9.5


six-node module, % of parameter sets: {'is_pulse': 25.12, 'is_damped_osc': 5.26, 'is_sustained_osc': 0.66}
certified limit cycles: 83 | four-node control, %: {'is_pulse': 21.77, 'is_damped_osc': 8.57, 'is_sustained_osc': 0.52}


## 13. Prognosis, essentiality and enrichment (Results 3.8)

Scripts: `scripts/10_survival_and_clinical/cox_models/10_survival_cox_hubs.R` → `scripts/03_ffl_census/11_ffl_module_membership.R` →
`cox_models/12_module_score_survival.R` → `13_circuit_level_3node_vs_6node.R`; `cox_models/15_external_metabric.R` (METABRIC); `analyses/six_node_pattern/S8/s8_circuit_survival.R`
→ `analyses/six_node_followups/Q8/q8_significant_bhat6_circuits.py` (six-node pattern circuits); `scripts/13_multiomics_and_screens/essentiality_and_druggability/A_depmap_essentiality.R`;
`scripts/04_motif_significance/08_motif_node_sets.py` → `scripts/12_enrichment_and_metabolism/over_representation/09_enrichment_ora.R`. GSEA of the FFL node sets: `scripts/12_enrichment_and_metabolism/gsea/01_build_ranked_lists.R` →
`02_build_genesets.R` → `07_ffl_class_gsea.R` → `10_final_tables_and_figures.R`. Metabolic hypotheses (Supplementary
Note S1): `scripts/12_enrichment_and_metabolism/metabolic/20_metabolic_genesets.R` → `21_metabolic_scores.R` → `22_metabolic_tumour_vs_normal.R` →
`23_metabolic_associations.R` → `24_metabolic_three_claims.R` … `30_metabolic_robustness.R`, and `scripts/12_enrichment_and_metabolism/metabolic/E1_metabolomics.R`,
`E2_pathway_gsva.R`, `E3_nrf2_keap1.R`, `E4_let7b_hk2_control.R`.

In [ ]:
metabric = pd.read_csv("results/external_METABRIC_cox.csv")
print("METABRIC:", len(metabric), "tests,", (metabric.q_value < 0.05).sum(), "with q < 0.05")
circuits = pd.read_csv("results/circuit_level_summary.csv")
print(circuits[["endpoint", "n_6node", "median_deltaC_6node_minus_core", "pct_6node_better_than_own_core"]].round(4).to_string(index=False))
pattern = pd.read_csv("analyses/six_node_followups/Q8/six_node_pattern_os_circuits_q_below_0.05.csv")
contains = pattern.apply(lambda r: r.astype(str).str.contains("STAT3|HIF1A").any(), axis=1)
print(len(pattern), "six-node pattern circuits add to their core (q < 0.05);", contains.sum(), "contain STAT3 or HIF1A")
depmap = pd.read_csv("results/multiomics/depmap_essentiality_nodes.csv").set_index("gene")
print(depmap.loc[["NFKB1", "COL1A1"], ["n_essential_breast", "n_breast_screened"]].to_string())
sets = ["3-miR", "3-TF", "3-Comp", "4-node", "5-node", "6-node"]
gsea = pd.read_csv("results/v4/gsea_ffl_class_comparison.csv").query("ranked_list == 'tumour_vs_normal'")
print(gsea[gsea.set.isin(sets)][["set", "size", "NES", "padj"]].to_string(index=False, float_format="%.3g"))
background = pd.read_csv("results/v4/gsea_ffl_class_network_background_test.csv").set_index("set")
only_higher = ["4node_not_3node", "5node_not_3node", "6node_not_3node", "higherorder_not_3node"]
print("mean |t| above the network's own nodes, upper-tail p:", background.loc[sets[:3] + only_higher, "p_vs_network_bg"].round(4).to_dict())
partners = pd.read_csv("results/v4/metabolic_claim_iii_agerage_partners.csv").set_index("partner")
print("AGE-RAGE score vs fibroblast and epithelial content, rho:",
      partners.loc[["CONTROL|CONTROL_CAF_FULL", "CONTROL|CONTROL_EPITHELIAL"], "rho"].round(3).to_dict())
folate = pd.read_csv("results/v4/metabolic_claim_ii_folate_tvn.csv")
folate = folate[(folate.set_id == "CURATED|CURATED_ANTIFOLATE_DETERMINANTS") & (folate.score == "GSVA")]
print("antifolate-resistance determinants, tumour vs normal:")
print(folate[["cohens_d", "p_limma"]].to_string(index=False, float_format="%.3g"))

METABRIC: 718 tests, 183 with q < 0.05
endpoint  n_6node  median_deltaC_6node_minus_core  pct_6node_better_than_own_core
      OS     1989                          0.0012                         52.2373
     PFI     1989                          0.0009                         51.5837
16 six-node pattern circuits add to their core (q < 0.05); 12 contain STAT3 or HIF1A
        n_essential_breast  n_breast_screened
gene                                         
NFKB1                    0                 53
COL1A1                   0                 53
   set  size   NES     padj
4-node   351 -1.85 3.92e-09
5-node   339 -1.86 3.92e-09
6-node   314 -1.89 2.91e-09
3-Comp   258 -1.91 9.46e-09
  3-TF   258 -1.91 9.46e-09
 3-miR   264 -1.91 1.19e-08
mean |t| above the network's own nodes, upper-tail p: {'3-miR': 0.0012, '3-TF': 0.0005, '3-Comp': 0.0003, '4node_not_3node': 0.9987, '5node_not_3node': 0.9964, '6node_not_3node': 0.9944, 'higherorder_not_3node': 0.9995}
AGE-RAGE score vs fibroblast a

## 14. Module-detection methods (Supplementary Note S7; Table S9)

Scripts, all in `scripts/14_module_detection/`: community detection `01_build_graph.R` → `02_cm2_mcl.R`, `03_clustermaker2_cyrest.py`,
`04_igraph_battery.R` → `05_analysis.R` … `12_figure.R`; MCODE `mcode.R` → `v7_mcode_validate.R` → `v7_mcode_run.R` →
`v7_mcode_characterise.R` → `v7_mcode_controls.R`; BioNet `bionet_01` … `bionet_07`; jActiveModules `jam_common.R` with
`jam_engine.cpp` → `jam_01` … `jam_05`. The files in `results/v7/` are for the deposited network, which still holds the
30 exemplar miRNA–miRNA edges; the MCODE control `drop_all_miRNA_miRNA` is the analysed network. The other methods were
re-run on the analysed network in `analyses/analysed_network_reruns/`: `v7/setup_sandbox.sh` →
`04_igraph_battery.R` and `10_subnetwork_sensitivity.R` in each sandbox → `v7/cd_audit.R` → `v7/seed_frequency.R`, and
`v7b/setup_sandbox.sh` → the BioNet and jActiveModules chains → `v7b/jam_converged_consensus.R` → `v7b/ora_modules.R`.

In [ ]:
table_s9 = pd.read_csv("supplementary_tables/TableS9_module_detection_outcomes.csv")
print(table_s9.iloc[:, :3].to_string(index=False, max_colwidth=70))
mcode = pd.read_csv("results/v7/mcode_17_controls_summary.csv").set_index("network")
print(mcode.loc[["drop_all_miRNA_miRNA", "as_published"], ["n_clusters", "top_size", "top_seed", "axis_found"]].to_string())
audit = "analyses/analysed_network_reruns/"
seeds = pd.read_csv(audit + "v7/cd_seed_frequency.csv")
print(seeds[["network", "algorithm", "seeds", "trio_together"]].to_string(index=False))
jam = pd.read_csv(audit + "v7b/jam_converged_consensus.csv")
print(jam[["network", "primary_seeds", "converged_seeds", "consensus_size", "COL1A1_in", "COL3A1_in",
           "miR29abc_converged_seeds_selecting"]].to_string(index=False))

                                  Method                              Modules (analysed network)                     Top module contains COL1A1, COL3A1 and hsa-miR-29a
                                   MCODE                                                      15 No; the top cluster (33 nodes, seeded at MMP2) contains neither col...
Community detection (38 igraph settings)                                            3–26 per run No; together in none of 14 modularity-optimising runs and in 18 of ...
                                  BioNet                                            1 (57 nodes) No; no collagen gene (NABA collagens 0 of 4; Naba et al. 2012) and ...
                          jActiveModules 1 consensus module (121 nodes; 9 of 10 seeds converged) No; contains COL1A1 but not COL3A1; hsa-miR-29a and hsa-miR-29c eac...
                      n_clusters  top_size top_seed  axis_found
network                                                        
drop_all_miRNA_miRNA          15

## 15. Tests with public perturbation data (Supplementary Note S1)

Scripts: `analyses/perturbation_tests/` (P1a TF knockdowns, P1b miR-29 gain and loss, P2 co-transcribed clusters, P3a/P3b
edge validation, P4 Borzoi, P5 cell atlases); the settings fixed in advance and their logged changes are in
`analyses/perturbation_tests/SETTINGS.md`, and every dataset found, with the reason for any exclusion, in
`analyses/perturbation_tests/DATASETS.tsv`. The per-gene differential-expression tables are not included; the
scripts rebuild them from GEO. Three sensitivity analyses (Notes S1 and S2) are in `analyses/perturbation_sensitivity/`:
`C1/c1_p1b_series_level.py` (P1b with the cell lines of each series averaged), `C2/c2_p3a_without_gse115646.py` (P3a
without GSE115646) and `C3/c3_p2_mouse_leave_one_series_out.py` (P2 mouse data, leaving out one series at a time).

In [ ]:
folder = "analyses/perturbation_tests/"
keys = {"P1a/p1a": "fibroblast lineage: k", "P1b/p1b": "(primary):", "P2/p2": "decision", "P3/p3a": "->",
        "P3/p3b": "TRRUST:", "P5/p5": "decision"}
for part, key in keys.items():
    print(part.split("/")[1].upper())
    for line in open(f"{folder}{part}_summary.txt"):
        if key in line and not line.startswith("contrast"):
            print("  " + line.strip()[:170])
checks = "analyses/perturbation_sensitivity/"
for part, key in {"C1/c1": "series level", "C2/c2": "strong without", "C3/c3": "leave one series out"}.items():
    print(part[:2])
    for line in open(f"{checks}{part}_summary.txt"):
        if key in line:
            print("  " + line.strip()[:170])

P1A
  ETS1 -> COL1A1, fibroblast lineage: k = 1; pooled log2FC +0.255 (95 % CI +0.072 to +0.439); fold change 1.194 (1.051-1.356); tau2 0; rule: INCONCLUSIVE [decision]
  ETS1 -> COL3A1, fibroblast lineage: k = 1; pooled log2FC +0.134 (95 % CI -0.036 to +0.305); fold change 1.098 (0.975-1.236); tau2 0; rule: SUPPORTED [decision]
  RELA -> COL1A1, fibroblast lineage: k = 5; pooled log2FC +0.216 (95 % CI -0.211 to +0.644); fold change 1.162 (0.864-1.563); tau2 0.195; rule: INCONCLUSIVE [descriptive]
  RELA -> COL3A1, fibroblast lineage: k = 5; pooled log2FC +0.243 (95 % CI -0.624 to +1.110); fold change 1.183 (0.649-2.158); tau2 0.915; rule: INCONCLUSIVE [descriptive]
P1B
  collagen pseudo-gene (primary): carcinoma mean +0.227; fibroblast minus carcinoma +0.888 (95 % CI +0.192 to +1.584), z-test p 0.0124; k = 9 with a value (fibroblast 4, ca
P2
  decision (human primary clusters, >=2 vs 1 members under network edges and TargetScan): NOT SUPPORTED
  same rule applied to the human secondar

## Not included

Patient-level data (TCGA-BRCA, METABRIC, CPTAC), licensed resources (OncoKB, COSMIC, KEGG, MSigDB) and large public
downloads (STRING, TargetScan, GEO series, model weights) are not redistributed. `README.md` lists their sources and
which scripts build the derived files from them.